# 🧬 Medical Imaging Dataset Acquisition & Processing Pipeline
This notebook is pre-configured to download, process, and prepare large medical datasets from Kaggle using Colab's high-performance cloud environment.

### 🛠️ Instructions:
1. Run the **Setup** cell to install Kaggle API and Kaggle Hub.
2. Upload your `kaggle.json` when prompted.
3. Configure your dataset selection and processing parameters.
4. Run the **Acquisition & Processing** cells.

In [ ]:
# @title ⚙️ Configuration
COVID_19 = True #@param {type:"boolean"}
NIH_CHEST = False #@param {type:"boolean"}
LUNG_DISEASE = False #@param {type:"boolean"}
SARS_MERS = False #@param {type:"boolean"}

TRAIN_SPLIT = 0.75 #@param {type:"number"}
VAL_SPLIT = 0.15 #@param {type:"number"}

APPLY_PREPROCESSING = True #@param {type:"boolean"}
APPLY_AUGMENTATION = True #@param {type:"boolean"}

SAVE_TO_DRIVE = True #@param {type:"boolean"}

In [ ]:
# @title 🚀 1. Setup & Installation
!pip install -q kaggle kagglehub opencv-python-headless albumentations
from google.colab import files
import os
import shutil
import kagglehub
import cv2
import numpy as np
from glob import glob

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file (from Kaggle Account settings).")
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        !mkdir -p ~/.kaggle
        !cp kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
        print("Kaggle API configured successfully.")
    else:
        print("No kaggle.json uploaded. Proceeding with kagglehub only.")

In [ ]:
# @title 📥 2. Dataset Acquisition
datasets = []
if COVID_19: datasets.append("tawsifurrahman/covid19-radiography-database")
if NIH_CHEST: datasets.append("nih-chest-xrays/data")
if LUNG_DISEASE: datasets.append("omkarmanohardalvi/lungs-disease-dataset-4-types")
if SARS_MERS: datasets.append("yazanqiblawey/sars-mers-xray-images-dataset")

os.makedirs('./raw_data', exist_ok=True)

for ds in datasets:
    print(f"--- Downloading {ds} using kagglehub ---")
    try:
        path = kagglehub.dataset_download(ds)
        print(f"Successfully downloaded to {path}")
        
        folder_name = ds.split('/')[-1]
        target_path = f"./raw_data/{folder_name}"
        
        if os.path.isdir(path):
            shutil.copytree(path, target_path, dirs_exist_ok=True)
        else:
            shutil.copy2(path, target_path)
        print(f"Moved files to {target_path}")
    except Exception as e:
        print(f"Error with kagglehub: {e}. Falling back to kaggle CLI...")
        !kaggle datasets download -d {ds} --unzip -p ./raw_data/{ds.split('/')[-1]}

In [ ]:
# @title 🛠️ 3. Preprocessing & Augmentation
import albumentations as A

def get_transforms():
    transforms = []
    if APPLY_PREPROCESSING:
        transforms.append(A.Resize(224, 224))
        transforms.append(A.Normalize())
    
    if APPLY_AUGMENTATION:
        transforms.append(A.Rotate(limit=15, p=0.5))
        transforms.append(A.HorizontalFlip(p=0.5))
        transforms.append(A.RandomBrightnessContrast(p=0.2))
        
    return A.Compose(transforms)

print("Pipeline configured with transforms:")
print(get_transforms())

def process_images():
    # This is a template function. In a real scenario, you would iterate over your files.
    print("Processing raw data...")
    # Example: 
    # images = glob('./raw_data/**/*.jpg', recursive=True)
    # for img_path in images: ...
    print("Processing complete.")

process_images()

In [ ]:
# @title 💾 4. Persistence (Google Drive)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Copying to Google Drive...")
    !cp -r ./raw_data /content/drive/MyDrive/MedicalDatasets_Final
    print("Backup complete at /content/drive/MyDrive/MedicalDatasets_Final")